In [1]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import requests
import json
from google import genai
import tiktoken

In [29]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
google_api_key =  os.getenv('GOOGLE_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
claude_api_key = os.getenv('CLAUDE_API_KEY')

In [6]:
openai = OpenAI()

In [4]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "How do you convert chat.completions.create to the newer Response API? I am thinking specifically of the openai.responses.create function."},
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
display(Markdown(response.choices[0].message.content))

To convert from `chat.completions.create` to the newer Response API using `openai.responses.create`, you need to understand that the Response API is designed to handle streaming responses and more flexible usage patterns.

Here's a general comparison and how to adapt your code:

### Original `chat.completions.create` Example:
```python
import openai

response = openai.ChatCompletion.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a joke."}
    ],
    max_tokens=50
)
print(response.choices[0].message.content)
```

### Converted to Response API (`openai.responses.create`):

In the newer API, you interact with the `openai.responses.create()` function, which is more flexible and supports streaming, to get responses (when used with a specific response type).

**Note:** The Response API currently involves a different flow, especially when handling streaming responses, and requires specifying the response type.

### Basic Example for Text Completion:

```python
import openai

# Prepare the request parameters
response = openai.responses.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a joke."}
    ],
    max_tokens=50,
    response_type="completion"  # Specify type of response if needed
)

# Read the generated text
print(response["choices"][0]["text"])
```

### Notes:
- `response_type` can be set to `"completion"` for the same kind of output as `ChatCompletion`.
- The response object structure may differ slightly, so check the API documentation for your specific needs.
- If you're streaming, you'd handle the response differently, typically iterating over chunks.

### Important:
- Make sure your openai SDK version supports `responses.create`.
- Always refer to the latest OpenAI API documentation for exact parameters and response handling.

---

**Summary:**
- Replace `openai.ChatCompletion.create()` with `openai.responses.create()`.
- Pass the same parameters (`model`, `messages`, etc.).
- Specify `response_type="completion"` if necessary.
- Extract the generated text from `response["choices"][0]["text"]` instead of `response.choices[0].message.content`.

---

If you provide a specific use case or code snippet, I can help tailor the conversion further!

In [9]:
response = openai.responses.create(
  model="gpt-5",
  input="How do you convert chat.completions.create to the newer Response API? I am thinking specifically of the openai.responses.create function."
)

In [14]:
display(Markdown(response.output_text))

Short version
- Replace chat.completions.create with responses.create.
- Pass your chat messages to input (you can keep them as role/content messages).
- Rename max_tokens to max_output_tokens.
- Read the text from response.output_text (or from response.output[0].content[...] for finer control).
- Tools/functions still go in tools; tool calls come back as content items of type "tool_use".

Minimal 1:1 conversion (no tools)

JavaScript (Node)
const OpenAI = require("openai");
const client = new OpenAI();

const resp = await client.responses.create({
  model: "gpt-4.1-mini",
  input: [
    { role: "system", content: "You are concise." },
    { role: "user", content: "Summarize the plot of Hamlet in one sentence." }
  ],
  max_output_tokens: 200,
  temperature: 0.7
});

// New way to get the text
console.log(resp.output_text);

Python
from openai import OpenAI
client = OpenAI()

resp = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Summarize the plot of Hamlet in one sentence."}
    ],
    max_output_tokens=200,
    temperature=0.7,
)

print(resp.output_text)

Tool/function calling (basic pattern)

- Define your tools the same way (type: "function").
- The model returns a tool request in response.output[0].content as items of type "tool_use".
- Call your function, then send a follow-up responses.create with a tool_result message.

JavaScript
const resp1 = await client.responses.create({
  model: "gpt-4.1-mini",
  input: [{ role: "user", content: "What's the weather in San Francisco right now?" }],
  tools: [
    {
      type: "function",
      function: {
        name: "get_weather",
        description: "Get the current weather",
        parameters: {
          type: "object",
          properties: { location: { type: "string" }, unit: { type: "string", enum: ["c", "f"] } },
          required: ["location"]
        }
      }
    }
  ]
});

// Find a tool call
const toolUse = resp1.output?.[0]?.content?.find(c => c.type === "tool_use");
if (toolUse) {
  // Call your function
  const result = await getWeather(toolUse.input.location, toolUse.input.unit);

  // Send tool result back to the model to get the final answer
  const resp2 = await client.responses.create({
    model: "gpt-4.1-mini",
    input: [
      { role: "user", content: "What's the weather in San Francisco right now?" },
      { role: "assistant", content: [toolUse] },
      {
        role: "tool",
        content: [
          {
            type: "tool_result",
            tool_use_id: toolUse.id,
            content: JSON.stringify(result)
          }
        ]
      }
    ],
    tools: [
      {
        type: "function",
        function: {
          name: "get_weather",
          parameters: {
            type: "object",
            properties: { location: { type: "string" }, unit: { type: "string", enum: ["c", "f"] } },
            required: ["location"]
          }
        }
      }
    ]
  });

  console.log(resp2.output_text);
} else {
  console.log(resp1.output_text);
}

Key parameter and shape changes
- messages -> input
  - You can pass an array of role/content messages (system, user, assistant, tool) just like before.
  - You can also pass a single string, or structured content parts (e.g., images).
- max_tokens -> max_output_tokens
- Accessing text
  - Old: completion.choices[0].message.content
  - New: response.output_text (convenient joined text), or drill into response.output[0].content where content items include:
    - { type: "output_text", text: "..." }
    - { type: "tool_use", id, name, input }
- Tools/functions
  - Define under tools with type: "function" (same JSON Schema).
  - tool_choice still supported (e.g., "auto", "required", or a specific function by name).
  - Parallel tool calls are supported; you may see multiple tool_use items.
- JSON output
  - Instead of response_format: { type: "json_object" }, prefer a schema:
    response_format: {
      type: "json_schema",
      json_schema: { name: "MySchema", schema: { ... }, strict: true }
    }
- Streaming
  - Old: stream: true with Chat Completions.
  - New: use client.responses.stream(...) and consume events (text deltas, tool calls, completion).
- Vision/multimodal
  - Provide input as content parts (e.g., { type: "input_text", text: "..." }, { type: "input_image", image_url: "..." }).

Quick before/after snapshot

Before (chat.completions.create)
const c = await openai.chat.completions.create({
  model: "gpt-4o-mini",
  messages: [{ role:"user", content:"Hello" }],
  max_tokens: 50
});
console.log(c.choices[0].message.content);

After (responses.create)
const r = await openai.responses.create({
  model: "gpt-4.1-mini",
  input: [{ role:"user", content:"Hello" }],
  max_output_tokens: 50
});
console.log(r.output_text);

If you share your current call, I can translate it line-by-line.

In [15]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [17]:
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [18]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-D68iIARIKmZCPjMYGCxt7XXtYbgtN',
 'object': 'chat.completion',
 'created': 1770355834,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: A day on Venus is longer than a year on Venus. It takes about 243 Earth days to rotate once, but only about 225 Earth days to orbit the Sun. Want another fun fact?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 564,
  'total_tokens': 575,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 512,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [19]:
fun_facts = openai.responses.create(
  model="gpt-5-nano",
  input="Tell me a fun fact"
)

display(Markdown(fun_facts.output_text))

Fun fact: Honey never really spoils. Archaeologists have found honey in ancient Egyptian tombs that’s still edible after thousands of years. Its longevity comes from low water content, acidic pH, and enzymes that produce hydrogen peroxide, which help prevent spoilage. Want another fun fact?

In [ ]:
#GEMINI

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

google_api_key = os.getenv("GOOGLE_API_KEY")

client = genai.Client()
response = client.models.generate_content(
    model="gemini-3-flash-preview", contents="Explain how AI works in a few words"
)
print(response.text)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


It finds patterns in data to make predictions.


In [28]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

display(Markdown(response.choices[0].message.content))

Here's a fun one:

**Octopuses have three hearts!**

*   Two of them pump blood to the gills, while the third pumps blood to the rest of the body. And when an octopus swims, its third heart stops beating!

In [ ]:
#OLLAMA
requests.get("http://localhost:11434").content # Ollama must be running, run ollama serve if it is not running

b'Ollama is running'

In [30]:
#Download the model

# !ollama pull llama3.2

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama') #api key is any thing, just needs to be set
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])
display(Markdown(response.choices[0].message.content))

In [ ]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

In [ ]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

In [ ]:
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"

In [2]:
#Tokenization

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [3]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [10]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    # {"role": "user", "content": "Hi! I'm Ed!"},
    # {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'I’m not sure what your name is based on our conversation so far. Could you please tell me your name?'

In [16]:
response = openai.responses.create(
  model="gpt-4.1-mini", input="My Name is Jack", store=True)

In [18]:
print(response)

Response(id='resp_084a2d849d46ccf2006986c2ed9d34819bb3363ec1abde0820', created_at=1770439405.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_084a2d849d46ccf2006986c2ee0250819b9a00843f0a8ef435', content=[ResponseOutputText(annotations=[], text='Hello Jack! How can I assist you today?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None), safety_identifier=None, service_tier='default', status='completed', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'), top_logprobs=0, truncation='disabled', usage=ResponseUsa

In [20]:
response_two = openai.responses.create(
  model="gpt-4.1-mini", input="Can you remember my name", previous_response_id=response.id)

In [23]:
print(response_two.output_text)

I can remember your name during this conversation, Jack! However, I won’t be able to retain it once our chat ends. How can I help you today?


In [24]:
response_two = openai.responses.create(
  model="gpt-4.1-mini", input="When you say you won't remember it after our chat ends what do you mean?", previous_response_id=response.id)
print(response_two.output_text)

When I say I won't remember your name or any details after our chat ends, I mean that I don’t retain information about individual users or conversations once the session is over. Each conversation with me is independent, and I don't store personal data or recall past interactions. This is designed to protect your privacy and confidentiality. If you start a new chat, you would need to provide any relevant information again. If you have any other questions about how I work, feel free to ask!


In [ ]:
response_two = openai.responses.create(
  model="gpt-4.1-mini", input="But during the duration of our conversation in this jupyter notebook, so long as i pass the response.id from the original response, you should remember it, right?", previous_response_id=response.id)


In this chat interface, I don't have memory between separate interactions unless context is maintained within the conversation itself. Passing a `response.id` back here doesn't grant me memory of previous responses because each message is processed independently without persistent state.

However, if you are working in a Jupyter notebook and want to maintain context or remember information during the notebook session, you could store responses and their IDs in a Python variable or data structure. Then you can refer back to this stored information when needed, and provide it explicitly as part of your prompts.

If you want, I can help you design a method to track and use such response IDs and their associated data inside your notebook. Would you like me to?


In [26]:
display(Markdown(response_two.output_text))

In this chat interface, I don't have memory between separate interactions unless context is maintained within the conversation itself. Passing a `response.id` back here doesn't grant me memory of previous responses because each message is processed independently without persistent state.

However, if you are working in a Jupyter notebook and want to maintain context or remember information during the notebook session, you could store responses and their IDs in a Python variable or data structure. Then you can refer back to this stored information when needed, and provide it explicitly as part of your prompts.

If you want, I can help you design a method to track and use such response IDs and their associated data inside your notebook. Would you like me to?

In [27]:
response_two = openai.responses.create(
  model="gpt-4.1-mini", input="Sure please design this in a markdown cell also, if i save the response each time to a variable, for persistence i need to use that variable.id to maintain persistence?", previous_response_id=response.id)


In [28]:
display(Markdown(response_two.output_text))

Sure, Jack! Here's your name "My Name is Jack" designed in a Markdown cell:

```markdown
# My Name is Jack
```

---

### Regarding your question about persistence and saving responses to a variable:

If you want to maintain persistence across multiple responses or cells in a notebook or an interactive environment, you generally save the response to a variable. However, a variable itself *is* the reference to the stored data.

- If you assign a response to a variable like `response = "My Name is Jack"`, then you can reuse `response` as much as you want.
- You do **not** need to use something like `variable.id` to maintain persistence.
- The `.id` property usually applies to objects like database records, API responses, or certain frameworks — it’s specific to those contexts.
- In general programming/scenarios like Python notebooks, saving the value to a variable *is* your persistence for that session.

---

If you have a specific system or framework in mind where `.id` is used, please share more details and I can help clarify!